In [ ]:
import numpy as np
import torch
from pytorch_msssim import SSIM
from tqdm.notebook import tqdm, trange

from models.Models import *
from models.MotionRNN import RNN
from data_metric_provider import set_seed, eval, DataProvider

from datetime import date

## Train

In [298]:
dp = DataProvider(cuda=0,
    grid_file='data3/grid.npy',
    files=[
        'data3/sentinel-HV_2014-10-25_2023-12-31.npy',
        'data3/sentinel-HH_2014-10-25_2023-12-31.npy',

        'data3/glorys-bottomT_2015-09-01_2023-10-24.npy',
        'data3/glorys-mlotst_2015-09-01_2023-10-24.npy',
        'data3/glorys-so_2015-09-01_2023-10-24.npy',
        'data3/glorys-thetao_2015-09-01_2023-10-24.npy',
        'data3/glorys-uo_2015-09-01_2023-10-24.npy',
        'data3/glorys-vo_2015-09-01_2023-10-24.npy',
        'data3/glorys-zos_2015-09-01_2023-10-24.npy',

        'data3/meteo-f_2014-01-01_2023-12-31.npy',
        'data3/meteo-P_2014-01-01_2023-12-31.npy',
        'data3/meteo-T_2014-01-01_2023-12-31.npy',
        'data3/meteo-u_2014-01-01_2023-12-31.npy',
        'data3/meteo-v_2014-01-01_2023-12-31.npy',
    ]
)
cell_areas = torch.Tensor(np.load('data3/cell_areas.npy'))

In [105]:
metrics={}
rmseday={}
rmsemonth={}
split_inds = [(date(2022+(i>1), (10+i)%12+1, 1) - date(2022, 10, 1)).days for i in range(11)]

In [ ]:
set_seed(0)
kwargs = dict(scale=1, w_w=6, i_w=0, e_w=3, starter='pers')

ch = dp.all_data.shape[1]+1
models = {
    'Persistence': lambda: Seq2seq(**kwargs),
    'Linear': lambda: Seq2seq(model=E(Lin(32, wn=False, bn=False)), **kwargs),
    'DMVFN': lambda: VFN(ch, **kwargs),
    'IAM4VP': lambda: S2SDU2(IAM(ch, **kwargs|{'w_w':9}), (64,64)),
    'Neural ODE': lambda: Seq2seq(model=MALI(DC(32,32)), outgate=Conv(32,1,1), **kwargs|{'scale':2}),
    'MotionRNN': lambda: Seq2seq(model=RNN(440, 200, ch, device=dp.DEVICE, layer_norm=0), **kwargs|{'scale':2}),
    'Vid-ODE': lambda: VidODE(ch, **kwargs),
    'UNet': lambda: FUNet(ch, **kwargs),
    'rUNet': lambda: Seq2seq(model=RUNet(32), **kwargs)
}
name = 'rUNet'
model = models[name]().to(dp.DEVICE)

LOAD_BATCH, TRAIN_BATCH = 1, 32
OPT_P = max(TRAIN_BATCH // LOAD_BATCH, 1)
LR = 2e-4 * LOAD_BATCH/32
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], LR)
sch = torch.optim.lr_scheduler.ExponentialLR(opt, 0.99)
train_aug, train, val, test = dp.get_loaders(LOAD_BATCH, augment=1, sic_augment=3, val_batch=2, **kwargs)
tr_loss, grads, tr_mse, vl_mse = [0], [0], [], []

In [ ]:
# train, plot definitions
weight = torch.Tensor([0.5]*model.i_w + [1]*model.e_w).to(dp.DEVICE)
ssim_loss = SSIM(data_range=1, size_average=True, channel=model.e_w)
ctm = dp.target_mask.to(dp.DEVICE)#[:,::model.scale,::model.scale]

def loss(pred, target, mask, weight=None, crop_loss=False):
    """seq2seq mse loss with weight for seq positions, shapes BTHW"""
    pred_ = (mask*pred*(target>0))
    target_ = (mask*target).expand(*pred.shape)
    if crop_loss:
        pred_ *= ctm
        target_ *= ctm

    se = (pred_ - target_)**2
    if len(pred.shape) == 6: # for multi-scale voxel flow
        se = (pred*(0.8**(8-torch.arange(pred.shape[0])))).sum(0)
    m1se = torch.mean(se, dim=(0,-2,-1))
    if weight is None:
        weight = torch.ones_like(m1se)
    wse = torch.sum(m1se * weight)/weight.sum()
    return wse + 0.2*(1 - ssim_loss(
        pred_.reshape(-1,*target.shape[1:])[:,-model.e_w:],
        target_.reshape(-1,*target.shape[1:])[:,-model.e_w:]))

def fit(epochs, crop_loss=False):
    bar = trange(len(tr_loss), epochs + 1)
    step = 0
    for epoch in bar:
        gn, tl = [], []

        model.train()
        subbar = tqdm(train_aug, leave=False)
        for X, y, m in subbar:
            l = loss(model(X), y, m, weight, crop_loss); tl.append(l.detach().cpu().item())
            l.backward()
            step += 1
            if step % OPT_P == 0:
                subgn = []
                for tag, value in model.named_parameters():
                    if value.grad is not None:
                        subgn.append(value.grad.norm().cpu().item())
                subgn = np.mean(subgn)/OPT_P
                gn.append(subgn); subbar.set_postfix(gn=subgn)
                opt.step(); opt.zero_grad()
        if sch: sch.step()

        tl = np.mean(tl)
        vm = eval(model, val, True, False, False, False, target_mask=dp.target_mask)['mse']
        tr = eval(model, test, True, False, False, False, target_mask=dp.target_mask)['mse']
        if not len(vl_mse) or vm < min(vl_mse):
            torch.save(model.state_dict(), f"weights/best.pt")
        torch.save(model.state_dict(), f"weights/last.pt")
        gn = np.mean(gn)

        tr_loss.append(tl)
        tr_mse.append(tr)
        vl_mse.append(vm)
        grads.append(gn)
        bar.set_postfix(l=tl, gn=gn, vm=vm)

In [ ]:
fit(200)

In [ ]:
model.load_state_dict(torch.load(f"weights/best.pt"))
metric = eval(model, test, target_mask=dp.target_mask, ms_ssim=True,
                iiee_at=[0.05,0.15,0.30,0.50,0.75], cell_areas=cell_areas)
metrics.loc[name] = metric
torch.save(model.state_dict(), f"weights/{name}.pt")
metric

  0%|          | 0/181 [00:00<?, ?it/s]

{'mse': 0.007592081608194572,
 '1-ssim': 0.008336842060089111,
 '1-ms_ssim': 0.0045871734619140625,
 'iiee@0.05': 0.08987987041473389,
 'iiee@0.15': 0.09982864558696747,
 'iiee@0.3': 0.08953135460615158,
 'iiee@0.5': 0.09797116369009018,
 'iiee@0.75': 0.06038009375333786}